[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Hyperparameter Search &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import randint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Search `max_depth` from 2 to 8 alone, with `folds`, and print `best_params_` and `best_score_`.

In [2]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=0)

depths = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": list(range(2, 9))}, cv=folds)
depths.fit(X_train, y_train)

print(depths.best_params_, round(depths.best_score_, 3))

{'max_depth': 4} 0.829


The first four lines rebuild the split and the folds as the worked examples did, since this notebook runs on its own. `range(2, 9)` stops before 9, so the search tries 2 to 8.

**2.** From `search.cv_results_`, the grid of depths and leaf sizes in the worked examples, print the mean score of every combination with `min_samples_leaf=20`, sorted by depth.

In [3]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=0)

grid = {"max_depth": [2, 3, 4, 5, 6, 8], "min_samples_leaf": [1, 5, 20]}
search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=folds).fit(X_train, y_train)
results = pd.DataFrame(search.cv_results_)

leaf_20 = results[results["param_min_samples_leaf"] == 20]
leaf_20[["param_max_depth", "mean_test_score"]].sort_values("param_max_depth").round(3)

,param_max_depth,mean_test_score
2,2,0.750
5,3,0.812
8,4,0.828
11,5,0.830
14,6,0.830
17,8,0.830


A boolean mask, from the **Pandas, Deep Dive** guide, keeps the six rows with a leaf size of 20.

**3.** Search the logistic pipeline's `C` over `[0.1, 1, 10]` with `scoring="f1"`, and print the winner.

In [4]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=0)

f1_search = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression()),
                         {"logisticregression__C": [0.1, 1, 10]}, cv=folds, scoring="f1")
f1_search.fit(X_train, y_train)

print(f1_search.best_params_, round(f1_search.best_score_, 3))
print("mean F1 for C of 0.1, 1 and 10:", pd.DataFrame(f1_search.cv_results_)["mean_test_score"].round(3).tolist())

{'logisticregression__C': 1} 0.812
mean F1 for C of 0.1, 1 and 10: [0.804, 0.812, 0.81]


`best_score_` is now a mean F1, since that is what the search was told to maximize. `C=1` wins, with 10 close behind at 0.810.

**4.** Run a `RandomizedSearchCV` of the tree with `n_iter=5` and `random_state=1`, and print its best settings and test accuracy.

In [5]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=0)

ranges = {"max_depth": randint(2, 12), "min_samples_leaf": randint(1, 40)}
few = RandomizedSearchCV(DecisionTreeClassifier(random_state=0), ranges, n_iter=5, cv=folds, random_state=1)
few.fit(X_train, y_train)

print(few.best_params_, round(few.score(X_test, y_test), 3))

{'max_depth': 7, 'min_samples_leaf': 16} 0.817


Another seed draws other combinations, and five draws cover the ranges more thinly than fifteen.

**5.** Estimate the logistic pipeline's search over `C` on the training stations with nested cross-validation, using `folds` inside and `StratifiedKFold(5, shuffle=True, random_state=1)` outside, and print the mean.

In [6]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=0)

inner = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression()),
                     {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}, cv=folds)
nested = cross_val_score(inner, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=1))

print(nested.round(3), "| mean:", round(nested.mean(), 3))

[0.817 0.833 0.85  0.872 0.867] | mean: 0.848


The outer folds split the 900 training stations, so the test set stays out of the estimate, and each fold runs the whole search on its own four fifths.

---

&#8592; **Back to:** [Hyperparameter Search](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/10-hyperparameter-search.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
